In [1]:
from vistiq.io import ImageWriterConfig, ImageWriter, ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.utils import ArrayIteratorConfig, check_device, resolve_futures 
from vistiq.core import Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import FuncProcessor, FuncProcessorConfig, PreprocessFlow, PreprocessFlowConfig, ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector

from prefect import flow, task
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait
from prefect.futures import resolve_futures_to_results

import stackview
import os
import copy
import numpy as np
import math
import logging

2026-06-04 16:11:13,613 - INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


# Configure logger and check availability of accelerators

In [2]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

2026-06-04 16:11:14,541 - INFO - Found mps device: Apple Metal (MPS)
2026-06-04 16:11:14,542 - INFO - Available Torch accelerators: mps


# Load image

In [3]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="Animal 1.lif"
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

scene_index = 0

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [4]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    #substack="C:3"
)
img, metadata = ImageLoader(ilc).run(path)

2026-06-04 16:11:14,647 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:11:14,704 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:11:14,708 - INFO - Loading image from: Animal 1.lif
2026-06-04 16:11:14,982 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-06-04 16:11:15,078 - INFO - Loaded image: Animal 1.lif scene=0 -> shape=(3, 93, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-06-04 16:11:15,080 - INFO - Loaded image with shape: (3, 93, 512, 512), dtype: uint8
2026-06-04 16:11:15,081 - INFO - Finished in state Completed()


In [5]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [6]:
ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
    ]
)
c_img, c_metadata = PreprocessFlow(ppcfg).run(img, metadata=metadata, workers=-1)
metadata, c_metadata

2026-06-04 16:11:15,146 - INFO - Setting up PreprocessFlow with Rescale,FuncProcessor,FuncProcessor,FuncProcessor,Rescale,FuncProcessor
2026-06-04 16:11:15,366 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-04 16:11:15,510 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-04 16:11:15,563 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:11:15,734 - INFO - HTTP Request: PATCH https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dbe-36e4-71a1-8000-868700404d2f "HTTP/1.1 204 N

({'scene_index': 0,
  'dim_order': 'CZYX',
  'axes': ['C', 'Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (3, 93, 512, 512),
  'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)})

2026-06-04 16:11:25,085 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-04 16:11:47,201 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-04 16:11:49,271 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-04 16:11:51,389 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-04 16:11:53,476 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/

In [7]:
stackview.slice(c_img)

# Segment Lobes 3D

In [8]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area", 
                range=(500, np.inf)
            )
        ),
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area_xz", 
                range=(100, np.inf)
            )
        ),
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area_yz", 
                range=(100, np.inf)
            )
        ),
        RangeFilter(
            RangeFilterConfig(
                attribute="aspect_ratio", 
                range=(0.5, 1.0)
            )
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.7,
    consensus_threshold=0.5,
)
lobe_labels, t_labels, t_masks, untiled, t_proj = TiledSegmentationFlow(tsfcfg).run(c_img, metadata=c_metadata, workers=2, verbose=0, config=tsfcfg)

2026-06-04 16:11:23,321 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-04 16:11:23,444 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-04 16:11:23,496 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:11:23,619 - INFO - HTTP Request: PATCH https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dbe-b619-7583-8000-6072b1dd05c9 "HTTP/1.1 204 No Content"
2026-06-04 16:11:23,774 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c

Using apple MPS device.


2026-06-04 16:11:29,454 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-06-04 16:11:29,454 - INFO - StackProcessor.run: received workers=2 (type: <class 'int'>)
2026-06-04 16:11:29,455 - INFO - Found mps device: Apple Metal (MPS)
2026-06-04 16:11:29,455 - INFO - Set MPS memory fraction to 1.0
2026-06-04 16:11:29,501 - INFO - Using ./embeddings/47b0a949323f1bcee23a

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-04 16:11:49,474 - INFO - Finished in state Completed()
2026-06-04 16:11:49,476 - INFO - Finished in state Completed()
2026-06-04 16:11:49,613 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'object_id', 'bbox']
2026-06-04 16:11:49,613 - INFO - StackProcessor.run: received workers=2 (type: <class 'int'>)
2026-06-04 16:11:49,637 - INFO - RegionAnalyzer: Applying scale: (-0.9999284782608696, 1.2003733855185907, 1.2003733855185907), labels.shape=(93, 399, 399), metadata['channel_names']=['Scrib', 'EdU', 'Dpn']
2026-06-04 16:11:49,668 - INFO - Identified 17 regions, return as dataframe
2026-06-04 16:11:49,669 - INFO -

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-04 16:11:51,477 - INFO - Finished in state Completed()
2026-06-04 16:11:51,478 - INFO - Finished in state Completed()
2026-06-04 16:11:51,648 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dbe-b619-7583-8000-6072b1dd05c9/set_state "HTTP/1.1 201 Created"
2026-06-04 16:11:52,081 - INFO - Finished in state Completed()


In [9]:
print (f"Unique labels (incl. background): {np.unique(lobe_labels)}")

Unique labels (incl. background): [0 1 2 3]


In [11]:
vlabels = np.concatenate(len(metadata["channel_names"])*[lobe_labels], axis=-1)
stackview.blend(vimg.astype("uint16"), vlabels.astype("uint64"), blend_factor=40)

# Analyze regions

In [12]:
racfg = RegionAnalyzerConfig(
    properties=["volume", "centroid", "cross_sectional_area", "cross_sectional_area_xz", "cross_sectional_area_yz", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=(-3,-2,-1)),
    output_type="dataframe"
)
ra = RegionAnalyzer(racfg)

lobe_measurements = ra.run(lobe_labels, metadata=c_metadata)

2026-06-04 16:11:57,570 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:11:57,640 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:11:57,902 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'object_id', 'volume', 'centroid', 'cross_sectional_area', 'cross_sectional_area_xz', 'cross_sectional_area_yz', 'bbox

In [13]:
lobe_measurements

,centroid-0,centroid-1,centroid-2,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,aspect_ratio,cross_sectional_area,cross_sectional_area_xz,cross_sectional_area_yz,volume,object_id
label,,,,,,,,,,,,,,,
1,-51.707332,112.782800,92.714330,21,0,47,91,506,466,0.734894,5425.694885,1683.687285,1348.588848,298819.791203,1f3cb33245324ed3becfca4505a99fcf
2,-41.252724,47.249384,75.077858,22,0,48,91,324,430,0.524400,2553.808517,116.442429,112.660077,10696.088587,b88df69a219142f99dd8dca62f6cc072
3,-53.097942,43.377914,57.897022,22,0,50,91,318,330,0.674796,4983.339731,1377.857053,1411.718115,273742.785396,1bb95534bb3f45a1aa47eac5174f6f88


# Labels to lobe masks

In [14]:
from vistiq.core import labels_to_masks

lobe_masks = labels_to_masks(lobe_labels)
brain_label = (lobe_labels>0).astype("uint16")

In [15]:
stackview.slice(np.concatenate([brain_label, *np.unstack(lobe_masks, axis=0)], axis=-1))

# Save label

In [16]:
c_metadata["channel_names"] = ["Lobe"]

b_metadata = copy.deepcopy(c_metadata)
b_metadata["channel_names"] = ["Brain"]

In [17]:
imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(lobe_labels, outpath, metadata=c_metadata)

imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(brain_label, outpath, metadata=b_metadata)


2026-06-04 16:12:08,320 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:12:08,382 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-04 16:12:08,525 - INFO - ImageWriter: using config: classname='Configurable' package='vistiq.core' version=None command_group=None path='.' format='tif' overwrite=True writer=None split_channels=False extension='tif'
2026-06-04 16:12:08,526 - INFO - Preparing to save image with metadata: {'scene_index': 0, 'dim_order': 'ZYX', 'axes': ['Z', 'Y', 'X'], 'channel_names': ['Lobe'], 'channel_axis': 0, 'shape': (93, 512, 512), 'dims': <Dimensions [Z: 93, Y: 512, X: 512]>, 'pixel_unit': 'um', 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477

# Segment Cells

In [18]:
from typing import Any, List, Tuple
from prefect import task, flow

In [19]:
@task(name="unwrap")
def unwrap_results(futures) -> tuple[Any,Any]:
    #print(len(futures), type(futures))
    a, b, = wait(futures)#.result()
    return a, b

@task
def unzip_results(results: List[Tuple]):
    # Converts [(1, 11), (2, 12)] into ([1, 2], [11, 12])
    return tuple(zip(*results))

In [20]:
# specify preprocessing config 
ppcfg = PreprocessFlowConfig(
    processors = [
        #DoGConfig(
        #    sigma_low=1, # 5, 
        #    sigma_high=2, #12, 
        #    normalize=True,
        #    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z focal plane and channel
        #)
    ]
)


In [21]:
# Specify segmentation config
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
    #gpu_fraction=0.3,
)

min_cell_radius = 2.0
max_cell_radius = 7.0
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area", 
                range=(np.pi*min_cell_radius**2, np.pi*max_cell_radius**2)
            )
        ),
    ]
)

sfcfg = SegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
)

In [22]:
import pandas as pd
import itertools

@flow
def analyze_cells(labels: list[np.ndarray], metadata: list[dict[str, Any]]) -> list[pd.DataFrame]:
    print ([l.shape for l in labels])
    print ([m["channel_names"] for m in metadata])
    racfg = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "cross_sectional_area_xz", "cross_sectional_area_yz", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=(-3,-2,-1)),
        output_type="dataframe"
    )
    ra = RegionAnalyzer(racfg)

    measurements = ra.run.map(labels, metadata=metadata)

    cdcfg = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    )
    label_index_combinations = list(itertools.combinations(range(len(labels)), 2))
    l1 = [labels[c[0]] for c in label_index_combinations]
    l2 = [labels[c[1]] for c in label_index_combinations]
    sn = [(metadata[c[0]]["channel_names"][0], metadata[c[1]]["channel_names"][0]) for c in label_index_combinations]
    print (sn)
    #for la1, la2, sna in zip(l1,l2,sn): 
    cim = CoincidenceDetector(cdcfg).run.map(l1, l2, stack_names=sn)
    return measurements
 

In [23]:
@flow
def full_pipeline(img, metadata=metadata):
    # preprocess
    preprocessed, preprocessed_metadata = PreprocessFlow(ppcfg).run(img, metadata=metadata)
    # split channels
    channels, channel_metadata = unstack_image(preprocessed, preprocessed_metadata, axis=metadata["channel_axis"], strict=False)
    # segment each channel separately
    cell_labels = SegmentationFlow(sfcfg).mapped_run(channels, metadata=channel_metadata)
    # analyze regions in each channel separately
    measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*channel_metadata, c_metadata, b_metadata])
    # make sure to resolve the futures to results
    return (resolve_futures(cell_labels), 
    resolve_futures(channel_metadata), 
    resolve_futures(measurements), 
    resolve_futures(preprocessed), 
    resolve_futures(channels))

In [24]:
cell_labels, cell_metadata, cell_measurements, preprocessed, channels = full_pipeline(img, metadata=metadata)

2026-06-04 16:12:17,850 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-04 16:12:18,022 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-04 16:12:18,178 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dc2-1f04-7450-8000-5cd466d7ead7/set_state "HTTP/1.1 201 Created"
2026-06-04 16:12:18,248 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dc2-1f04-7450-8000-5cd466d7ead7 "HTTP/1.1 200 OK"
2026-06-04 16:12:18,300 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe0

Using apple MPS device.
Using apple MPS device.
Using apple MPS device.


2026-06-04 16:12:31,516 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-06-04 16:12:31,518 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-04 16:12:31,519 - INFO - Found mps device: Apple Metal (MPS)
2026-06-04 16:12:31,519 - INFO - Set MPS memory fraction to 1.0
2026-06-04 16:12:31,531 - INFO - Using ./embeddings/040b65c384168bc8f1a

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-04 16:13:21,233 - INFO - Finished in state Completed()
2026-06-04 16:13:21,303 - INFO - Finished in state Completed()
2026-06-04 16:13:21,305 - INFO - Finished in state Completed()
2026-06-04 16:13:21,350 - INFO - Finished in state Completed()
2026-06-04 16:13:21,395 - INFO - Finished in state Completed()
2026-06-04 16:13:21,559 - INFO - DEBUG: entered Relabeler.run
2026-06-04 16:13:22,871 - INFO - Finished in state Completed()
2026-06-04 16:13:22,906 - INFO - Finished in state Completed()
2026-06-04 16:13:23,058 - INFO - DEBUG: entered Relabeler.run
2026-06-04 16:13:25,851 - INFO - Finished in state Completed()
2026-06-04 16:13:25,895 - INFO - Creating RegionAnalyzer for region filter with properties: ['label', 'object_id', 'centroid', 'cross_sectional_area']
2026-06-04 16:13:26,133 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_s

...

2026-06-04 16:13:26,800 - INFO - Finished in state Completed()
2026-06-04 16:13:26,870 - INFO - Finished in state Completed()
2026-06-04 16:13:26,872 - INFO - Finished in state Completed()
2026-06-04 16:13:29,425 - INFO - Finished in state Completed()
2026-06-04 16:13:29,458 - INFO - Creating RegionAnalyzer for region filter with properties: ['label', 'object_id', 'centroid', 'cross_sectional_area']
2026-06-04 16:13:29,694 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='list' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'object_id', 'centroid', 'cross_sectional_area']
2026-06-04 16:13:29,695 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-04 16:

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-04 16:13:30,321 - INFO - Finished in state Completed()
2026-06-04 16:13:30,390 - INFO - Finished in state Completed()
2026-06-04 16:13:30,391 - INFO - Finished in state Completed()
2026-06-04 16:13:30,623 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dc2-5641-7f27-8000-5917c3b7a2c1/set_state "HTTP/1.1 201 Created"
2026-06-04 16:13:31,181 - INFO - Finished in state Completed('All states completed.')
2026-06-04 16:13:31,324 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/task_runs/ "HTTP/1.1 201 Created"
2026-06-04 16:13:31,426 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/filter "HTTP/1.1 200 OK"
2026-06-04 16:13:31,517 - INFO - HTTP Request: P

[(93, 512, 512), (93, 512, 512), (93, 512, 512), (93, 512, 512), (93, 512, 512)]
[['Scrib'], ['EdU'], ['Dpn'], ['Lobe'], ['Brain']]
[('Scrib', 'EdU'), ('Scrib', 'Dpn'), ('Scrib', 'Lobe'), ('Scrib', 'Brain'), ('EdU', 'Dpn'), ('EdU', 'Lobe'), ('EdU', 'Brain'), ('Dpn', 'Lobe'), ('Dpn', 'Brain'), ('Lobe', 'Brain')]


2026-06-04 16:13:32,485 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/v2/concurrency_limits/increment-with-lease "HTTP/1.1 200 OK"
2026-06-04 16:13:32,542 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'object_id', 'volume', 'centroid', 'cross_sectional_area', 'cross_sectional_area_xz', 'cross_sectional_area_yz', 'bbox', 'aspect_ratio']
2026-06-04 16:13:32,542 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIte

In [25]:
measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*cell_metadata, c_metadata, b_metadata])

2026-06-04 16:13:54,949 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-04 16:13:55,078 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-04 16:13:55,226 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dc8-302d-7d6d-8000-848e67c926a8/set_state "HTTP/1.1 201 Created"
2026-06-04 16:13:55,303 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a21dc8-302d-7d6d-8000-848e67c926a8 "HTTP/1.1 200 OK"
2026-06-04 16:13:55,352 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe0

[(93, 512, 512), (93, 512, 512), (93, 512, 512), (93, 512, 512), (93, 512, 512)]
[['Scrib'], ['EdU'], ['Dpn'], ['Lobe'], ['Brain']]
[('Scrib', 'EdU'), ('Scrib', 'Dpn'), ('Scrib', 'Lobe'), ('Scrib', 'Brain'), ('EdU', 'Dpn'), ('EdU', 'Lobe'), ('EdU', 'Brain'), ('Dpn', 'Lobe'), ('Dpn', 'Brain'), ('Lobe', 'Brain')]


2026-06-04 16:13:55,629 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'object_id', 'volume', 'centroid', 'cross_sectional_area', 'cross_sectional_area_xz', 'cross_sectional_area_yz', 'bbox', 'aspect_ratio']
2026-06-04 16:13:55,728 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-04 16:13:55,733 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_s

In [26]:
stackview.slice(np.concatenate(cell_labels, axis=-1))

In [27]:
for m in cell_measurements:
    print (m.describe())

       centroid-0  centroid-1  centroid-2      bbox-0      bbox-1      bbox-2  \
count  457.000000  457.000000  457.000000  457.000000  457.000000  457.000000   
mean   -47.874341   82.519214   70.304893   46.910284  261.398249  219.759300   
std     22.028326   45.311538   38.908466   21.961855  151.358943  129.767061   
min    -90.993492    0.724177    0.928601    0.000000    0.000000    0.000000   
25%    -71.994850   39.348168   37.869077   28.000000  116.000000  109.000000   
50%    -43.079992   89.887180   72.977706   43.000000  287.000000  228.000000   
75%    -27.997997  124.783228  104.653415   71.000000  399.000000  335.000000   
max     -0.999928  150.199135  152.494078   91.000000  488.000000  502.000000   

           bbox-3      bbox-4      bbox-5  aspect_ratio  cross_sectional_area  \
count  457.000000  457.000000  457.000000    281.000000            457.000000   
mean    49.792123  289.404814  250.142232      0.390197             45.185064   
std     22.150663  150.3636

# Hierarchical label decomposition

In [26]:
from vistiq.analysis import CoincidenceDetector, CoincidenceDetectorConfig

cdcfg = CoincidenceDetectorConfig(
    method="ios",
    iterator_config=ArrayIteratorConfig(slice_def=()),
    mode="outline",
)
df = CoincidenceDetector(cdcfg).run(lobe_labels, cell_labels[2], stack_names=["Lobe", "Dpn"])   

2026-06-03 18:42:02,410 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-03 18:42:02,456 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-03 18:42:02,973 - INFO - Running CoincidenceDetector with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='list' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None output=['score', 'above_threshold'] method='ios' mode='outline' threshold=0.5
2026-06-03 18:42:02,974 - INFO - StackProcessor.run: received workers=-

In [27]:
df[1].values()

dict_values([       Dpn +  ios Dpn +
label                  
1       True   1.000000
2      False   0.079227
3       True   1.000000,        Lobe +  ios Lobe +
label                    
1       False         0.0
2       False         0.0
3       False         0.0
4       False         0.0
5       False         0.0
...       ...         ...
200      True         1.0
201      True         1.0
202      True         1.0
203      True         1.0
204      True         1.0

[204 rows x 2 columns]])

In [ ]:
from vistiq.analysis.coincidence import box_iou_batch_3d, labels_iou_batch_3d, labels_iou_batch_3d_torch
from joblib import Parallel, delayed

bbox_cols = [b for b in lobe_measurements.columns.to_list() if "bbox" in b]
lobe_bboxes = lobe_measurements[bbox_cols].to_numpy()
threshold = 0.5
cell_boxes_list = [cm[bbox_cols].to_numpy() for cm in cell_measurements]
#ios = Parallel(verbose=10)(delayed(box_iou_batch_3d)(lobe_bboxes, cb, overlap_metric="IOS") for cb in cell_boxes_list)
ios = Parallel(verbose=10, prefer="threads", n_jobs=1)(delayed(labels_iou_batch_3d_torch)(lobe_labels, cl, overlap_metric="IOS", dense_pair_fraction=0.75) for cl in cell_labels)
#for cl, cm in zip(cell_labels,cell_measurements):
#    cell_bboxes = cm[bbox_cols].to_numpy()
#    #ios = labels_iou_batch_3d(lobe_labels, cl, overlap_metric="IOS")
#    ios = box_iou_batch_3d(lobe_bboxes, cell_bboxes, overlap_metric="IOS")
#    print (ios.shape, len(np.argwhere(ios>threshold)), len(np.argwhere(ios<=threshold)))
for i in ios:
    print (i.shape)

# View in Napari

In [26]:
import napari
viewer = napari.Viewer()

In [27]:
scale = metadata["physical_pixel_sizes"]
channel_colors = ("green", "blue", "red")
nimg = img#np.expand_dims(img, axis=0)

# add brain label
viewer.add_labels(brain, name="Brain", scale=scale)

# add lobe labels
for ch, l, m in zip(metadata["channel_names"],cell_labels, cell_measurements):
    new_m = m.copy().reset_index()
    background = pd.DataFrame({c: [0] if c=="label" else [0.0] for c in new_m.columns.to_list()})
    new_m = pd.concat([background, new_m], ignore_index=True)
    # print (new_m)
    viewer.add_labels(l, name=f"{ch}-Labels", features=new_m, scale=scale)

# add cell labels for each channel
ch_images, ch_metadata = unstack_image(img, metadata=metadata, axis="C", strict=False)
for name, c_img, color in zip(metadata["channel_names"], ch_images, channel_colors):
    viewer.add_image(c_img, name=f"{name}", scale=scale, colormap=color, blending="additive")
    


2026-06-02 23:44:39,924 - INFO - Unstacked image data along C axis with index 0. Data shapes: [(93, 512, 512), (93, 512, 512), (93, 512, 512)]


In [ ]:
dl1 = viewer.layers["Dpn-Labels-Lobe 1"]
dl2 = viewer.layers["Dpn-Labels-Lobe 2"]
print (np.intersect1d(np.unique(dl1.data), np.unique(dl2.data)))
